In [7]:
!python -m venv venv

In [9]:
!venv\Scripts\activate

!ollama pull phi3:mini
!ollama pull qwen2.5-coder:7b

In [1]:
!ollama pull deepseek-coder:6.7b

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest 
pulling 59bb50d8116b:   0% ▕                  ▏ 691 KB/3.8 GB                  pulling manifest 
pulling 59bb50d8116b:   0% ▕                  ▏ 1.5 MB/3.8 GB                  pulling manifest 
pulling 59bb50d8116b:   0% ▕                  ▏ 2.9 MB/3.8 GB                  pulling manifest 
pulling 59bb50d8116b:   0% ▕                  ▏ 3.5 MB/3.8 GB                  pulling manifest 
pulling 59bb50d8116b:   0% ▕                  ▏ 4.8 MB/3.8 GB                  pulling manifest 
pulling 59bb50d8116b:   0% ▕             

In [2]:
!ollama list

NAME                   ID              SIZE      MODIFIED       
deepseek-coder:6.7b    ce298d984115    3.8 GB    27 seconds ago    
qwen2.5-coder:1.5b     d7372fd82851    986 MB    4 hours ago       
phi3:mini              4f2222927938    2.2 GB    30 hours ago      
glm-5.1:cloud          59472abf9d0a    -         12 days ago       
qwen3:4b               359d7dd4bcda    2.5 GB    2 weeks ago       


installer dépendences

In [ ]:
%pip install pandas sqlalchemy pymysql requests sentence-transformers faiss-cpu sqlglot

In [11]:
import pymysql
print("PyMySQL installed")

PyMySQL installed


change le chemain après: (normal terminal)
    cd C:\Users\Nouhaila\Desktop\test_db-master
    C:\xampp\mysql\bin\mysql.exe -u root employee
    source employees.sql;
FOR TEST : 
    SHOW TABLES;
    SELECT COUNT(*) FROM employees;

Connecté Mysql

In [11]:
from sqlalchemy import create_engine
import pandas as pd

DATABASE_URL = (
    "mysql+pymysql://root:@localhost:3306/employees"
)

engine = create_engine(DATABASE_URL)

print("MySQL connected")

MySQL connected


In [12]:
import pandas as pd

tables = pd.read_sql("SHOW TABLES;", engine)

tables

,Tables_in_employees
0,current_dept_emp
1,departments
2,dept_emp
3,dept_emp_latest_date
4,dept_manager
5,employees
6,salaries
7,titles


Auto schéma extracteur

In [13]:
from sqlalchemy import inspect

inspector = inspect(engine)

schema_dict = {}

for table in inspector.get_table_names():

    columns = inspector.get_columns(table)

    schema_dict[table] = [
        column["name"]
        for column in columns
    ]

schema_dict

{'departments': ['dept_no', 'dept_name'],
 'dept_emp': ['emp_no', 'dept_no', 'from_date', 'to_date'],
 'dept_manager': ['emp_no', 'dept_no', 'from_date', 'to_date'],
 'employees': ['emp_no',
  'birth_date',
  'first_name',
  'last_name',
  'gender',
  'hire_date'],
 'salaries': ['emp_no', 'salary', 'from_date', 'to_date'],
 'titles': ['emp_no', 'title', 'from_date', 'to_date']}

In [14]:
schema_metadata = []

for table, columns in schema_dict.items():

    schema_metadata.append({
        "table": table,
        "columns": columns
    })

schema_metadata

[{'table': 'departments', 'columns': ['dept_no', 'dept_name']},
 {'table': 'dept_emp',
  'columns': ['emp_no', 'dept_no', 'from_date', 'to_date']},
 {'table': 'dept_manager',
  'columns': ['emp_no', 'dept_no', 'from_date', 'to_date']},
 {'table': 'employees',
  'columns': ['emp_no',
   'birth_date',
   'first_name',
   'last_name',
   'gender',
   'hire_date']},
 {'table': 'salaries',
  'columns': ['emp_no', 'salary', 'from_date', 'to_date']},
 {'table': 'titles', 'columns': ['emp_no', 'title', 'from_date', 'to_date']}]

In [15]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded


shema text

In [16]:
schema_texts = []

for item in schema_metadata:

    text = f"""
    Table: {item['table']}
    Columns: {", ".join(item['columns'])}
    """

    schema_texts.append(text)

schema_texts

['\n    Table: departments\n    Columns: dept_no, dept_name\n    ',
 '\n    Table: dept_emp\n    Columns: emp_no, dept_no, from_date, to_date\n    ',
 '\n    Table: dept_manager\n    Columns: emp_no, dept_no, from_date, to_date\n    ',
 '\n    Table: employees\n    Columns: emp_no, birth_date, first_name, last_name, gender, hire_date\n    ',
 '\n    Table: salaries\n    Columns: emp_no, salary, from_date, to_date\n    ',
 '\n    Table: titles\n    Columns: emp_no, title, from_date, to_date\n    ']

In [17]:
schema_embeddings = embedding_model.encode(
    schema_texts,
    convert_to_tensor=False
)

print("Embeddings created")

Embeddings created


In [18]:
import faiss
import numpy as np

embedding_matrix = np.array(schema_embeddings).astype("float32")

index = faiss.IndexFlatL2(
    embedding_matrix.shape[1]
)

index.add(embedding_matrix)

print("FAISS index ready")

FAISS index ready


Database TEST

In [19]:
import time
import pandas as pd

start = time.time()

df = pd.read_sql(
    "SELECT * FROM employees LIMIT 5;",
    engine
)

end = time.time()

print(df)

print(f"\nExecution time: {end - start:.2f} sec")

   emp_no  birth_date first_name last_name gender   hire_date
0   10001  1953-09-02     Georgi   Facello      M  1986-06-26
1   10002  1964-06-02    Bezalel    Simmel      F  1985-11-21
2   10003  1959-12-03      Parto   Bamford      M  1986-08-28
3   10004  1954-05-01  Chirstian   Koblick      M  1986-12-01
4   10005  1955-01-21    Kyoichi  Maliniak      M  1989-09-12

Execution time: 0.02 sec


In [36]:
user_question = "Show the department with the highest average salary"

In [37]:
import time

query = "Show employees current salaries"

start = time.time()

query_embedding = embedding_model.encode([query])

end = time.time()

print(query_embedding.shape)

print(f"\nEmbedding time: {end - start:.4f} sec")

(1, 384)

Embedding time: 0.0453 sec


In [38]:
import numpy as np
import time

query = "Show employees current salaries"

start = time.time()

query_embedding = embedding_model.encode([query])

D, I = index.search(
    np.array(query_embedding).astype("float32"),
    k=3
)

end = time.time()

print("Relevant schema:\n")

for idx in I[0]:
    print(schema_texts[idx])

print(f"\nRetrieval time: {end - start:.4f} sec")

Relevant schema:


    Table: salaries
    Columns: emp_no, salary, from_date, to_date
    

    Table: employees
    Columns: emp_no, birth_date, first_name, last_name, gender, hire_date
    

    Table: dept_manager
    Columns: emp_no, dept_no, from_date, to_date
    

Retrieval time: 0.1069 sec


In [39]:
mini_schema = ""

for idx in I[0]:

    mini_schema += schema_texts[idx]
    mini_schema += "\n"

print(mini_schema)

print("\nCharacters:", len(mini_schema))


    Table: salaries
    Columns: emp_no, salary, from_date, to_date
    

    Table: employees
    Columns: emp_no, birth_date, first_name, last_name, gender, hire_date
    

    Table: dept_manager
    Columns: emp_no, dept_no, from_date, to_date
    


Characters: 254


In [40]:
business_rules = """

IMPORTANT BUSINESS RULES:

- Current salaries are rows where to_date = '9999-01-01'

"""

In [47]:
relationship_rules = """

IMPORTANT RELATIONSHIPS:

- dept_emp contains all employee department assignments
- dept_manager contains only department managers

- Use dept_emp for employee department analysis
- Use dept_manager only for manager-specific questions

Examples:
- employees per department -> use dept_emp
- average employee salary by department -> use dept_emp
- current managers -> use dept_manager

"""

LLM speed

In [48]:
import requests

prompt = f"""
You are a MySQL SQL translator.

Use ONLY the schema provided below.

Database schema:

{mini_schema}

{business_rules}

{relationship_rules}

IMPORTANT:
- Return ONLY SQL
- No markdown
- No explanations
- Use ONLY exact column names from schema

Question:
{user_question}

SQL:
"""

start = time.time()

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "deepseek-coder:6.7b",
        "prompt": prompt,
        "stream": False,
        "temperature": 0,
        "num_predict": 80
    }
)

end = time.time()

data = response.json()

print(data["response"])

print(f"\nGeneration time: {end - start:.2f} sec")

SELECT dept_no, AVG(salary) AS avg_salary 
FROM salaries 
INNER JOIN employees ON salaries.emp_no = employees.emp_no 
INNER JOIN dept_manager ON employees.emp_no = dept_manager.emp_no
WHERE salaries.to_date = '9999-01-01' AND dept_manager.to_date = '9999-01<｜begin▁of▁sentence｜>23456789-01-01' 
GROUP BY dept_no 
ORDER BY avg_salary DESC LIMIT 1;


Generation time: 81.11 sec


Embendding speed

Test vector search

Test mini schema size

Test SQL generation

SQL cleanser

In [50]:
import re

sql_query = response.json()["response"]

# remove markdown
sql_query = re.sub(r"```sql", "", sql_query)
sql_query = re.sub(r"```", "", sql_query)

# remove special tokens
sql_query = re.sub(r"<\|.*?\|>", "", sql_query)

# remove weird unicode artifacts
sql_query = re.sub(r"▁", "", sql_query)

# normalize spaces
sql_query = re.sub(r"\s+", " ", sql_query)

sql_query = sql_query.strip()

# keep first query only
sql_query = sql_query.split(";")[0] + ";"

print(sql_query)

SELECT dept_no, AVG(salary) AS avg_salary FROM salaries INNER JOIN employees ON salaries.emp_no = employees.emp_no INNER JOIN dept_manager ON employees.emp_no = dept_manager.emp_no WHERE salaries.to_date = '9999-01-01' AND dept_manager.to_date = '9999-01<｜beginofsentence｜>23456789-01-01' GROUP BY dept_no ORDER BY avg_salary DESC LIMIT 1;
